<a href="https://colab.research.google.com/github/loucasty-cell/SVD/blob/main/ImageDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import cv2
import numpy as np
import scipy.stats
import scipy.signal
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

# =====================================================================
# 1. PRE-TRAINED DEEP LEARNING MODEL ENGINE
# =====================================================================

class PretrainedDeepfakeClassifier:
    """
    Loads pre-trained weights for ViT (Vision Transformer) fine-tuned on
    millions of AI-generated vs Real images.
    """
    def __init__(self, model_name: str = "Organika/sdxl-detector"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[+] Loading pre-trained weights from '{model_name}' on {self.device.upper()}...")

        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def predict_probability(self, pil_image: Image.Image) -> float:
        """
        Returns AI probability score between 0.0 and 1.0 using the pre-trained ViT model.
        """
        inputs = self.processor(images=pil_image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)[0]

        # Map label names to probability score
        labels = self.model.config.id2label
        ai_prob = 0.0
        for idx, label in labels.items():
            if "ai" in label.lower() or "fake" in label.lower() or "synthetic" in label.lower():
                ai_prob = probs[idx].item()
                break
            elif "real" in label.lower() or "human" in label.lower():
                ai_prob = 1.0 - probs[idx].item()

        return float(ai_prob)


# =====================================================================
# 2. SVD & SIGNAL PROCESSING FORENSIC ENGINE
# =====================================================================

class SVDSignalEngine:
    """
    Computes SVD singular value decay, Frobenius norm error, and 2D-FFT metrics.
    """
    @staticmethod
    def _to_gray_float64(pil_image: Image.Image) -> np.ndarray:
        img_arr = np.array(pil_image.convert("RGB"))
        ycbcr = cv2.cvtColor(img_arr, cv2.COLOR_RGB2YCrCb)
        return ycbcr[:, :, 0].astype(np.float64)

    def analyze_svd_spectrum(self, img_gray: np.ndarray) -> dict:
        """Fits power-law decay exponent gamma to singular values: S_k ~ k^-gamma"""
        img_centered = img_gray - np.mean(img_gray)
        M, N = img_centered.shape
        max_rank = min(M, N)

        S = np.linalg.svd(img_centered, compute_uv=False)
        if S[0] < 1e-9:
            return {"gamma": 1.25, "is_ai_svd": False}

        norm_s = S / S[0]
        k_min = max(2, int(0.05 * max_rank))
        k_max = max(k_min + 5, int(0.30 * max_rank))

        k_indices = np.arange(k_min, k_max + 1, dtype=np.float64)
        s_slice = np.clip(norm_s[k_min - 1 : k_max], 1e-12, None)

        slope, _, _, _, _ = scipy.stats.linregress(np.log(k_indices), np.log(s_slice))
        gamma = -slope

        return {"gamma": float(gamma), "is_ai_svd": bool(gamma < 0.95)}

    def compute_frobenius_residual(self, img_gray: np.ndarray, rank_ratio: float = 0.10) -> dict:
        """Calculates Frobenius error ratio R_f = ||I - I_k||_F / ||I||_F"""
        M, N = img_gray.shape
        k = max(1, int(rank_ratio * min(M, N)))
        img_centered = img_gray - np.mean(img_gray)

        U, S, Vt = np.linalg.svd(img_centered, full_matrices=False)
        I_k = np.dot(U[:, :k] * S[:k], Vt[:k, :])
        E = img_centered - I_k

        norm_I = np.linalg.norm(img_centered, "fro")
        norm_E = np.linalg.norm(E, "fro")
        R_f = float(norm_E / (norm_I + 1e-9))

        return {"R_f": R_f, "is_ai_frobenius": bool(R_f > 0.08)}


# =====================================================================
# 3. HYBRID FORENSIC DETECTOR ENSEMBLE
# =====================================================================

class HybridAIDetector:
    """
    Ensemble system fusing pre-trained ViT neural embeddings with
    SVD singular value spectrum metrics.
    """
    def __init__(self, vit_model_name: str = "Organika/sdxl-detector"):
        self.dl_engine = PretrainedDeepfakeClassifier(model_name=vit_model_name)
        self.svd_engine = SVDSignalEngine()

    def predict(self, image_path: str) -> dict:
        pil_img = Image.open(image_path).convert("RGB")
        img_gray = self.svd_engine._to_gray_float64(pil_img)

        # 1. Pre-trained Neural Classifier Score
        dl_ai_prob = self.dl_engine.predict_probability(pil_img)

        # 2. SVD Forensic Signal Metrics
        svd_res = self.svd_engine.analyze_svd_spectrum(img_gray)
        frob_res = self.svd_engine.compute_frobenius_residual(img_gray)

        # Calculate SVD Anomaly Factor (0.0 to 1.0)
        svd_score = 0.0
        if svd_res["is_ai_svd"]:
            svd_score += 0.6
        if frob_res["is_ai_frobenius"]:
            svd_score += 0.4

        # 3. Hybrid Weighted Fusion (70% Neural ViT + 30% Physical SVD)
        final_probability = (0.70 * dl_ai_prob) + (0.30 * svd_score)
        is_ai = final_probability >= 0.50

        return {
            "is_ai_predicted": is_ai,
            "final_ai_probability": round(final_probability * 100, 2),
            "dl_pretrained_confidence": round(dl_ai_prob * 100, 2),
            "svd_decay_gamma": svd_res["gamma"],
            "frobenius_residual_R_f": frob_res["R_f"],
            "svd_flagged": svd_res["is_ai_svd"] or frob_res["is_ai_frobenius"],
        }


# =====================================================================
# EXECUTION DEMO
# =====================================================================

if __name__ == "__main__":
    # Initialize using the verified Organika SDXL detector
    detector = HybridAIDetector(vit_model_name="Organika/sdxl-detector")

    # Replace with your test image path
    TEST_IMAGE_PATH = "sample_test.jpg"

    try:
        results = detector.predict(TEST_IMAGE_PATH)
        print("\n" + "=" * 50)
        print("HYBRID SVD + PRE-TRAINED AI DETECTION REPORT")
        print("=" * 50)
        print(f"Final AI Probability  : {results['final_ai_probability']}%")
        print(f"Classification        : {'[AI-GENERATED]' if results['is_ai_predicted'] else '[REAL PHOTO]'}")
        print("-" * 50)
        print(f"Pre-trained ViT Score : {results['dl_pretrained_confidence']}%")
        print(f"SVD Decay Slope (γ)   : {results['svd_decay_gamma']:.4f}")
        print(f"Frobenius Residual R_f: {results['frobenius_residual_R_f']:.4f}")
        print("=" * 50)
    except FileNotFoundError:
        print(f"[!] Please place a test image at '{TEST_IMAGE_PATH}' and re-run.")

[+] Loading pre-trained weights from 'Organika/sdxl-detector' on CPU...


Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

[!] Please place a test image at 'sample_test.jpg' and re-run.
